# Titanium alloy aging: reproduce the ML analysis
Download this GitHub repository as a ZIP, upload it below, and run the cells from top to bottom. The last cells display the newly calculated scores and plots, then download the new results.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Upload Titanium-Alloy-Aging-ML.zip'
PACKAGE_ZIP = zip_names[0]
print('Uploaded:', PACKAGE_ZIP)


In [ ]:
import zipfile, shutil
from pathlib import Path
WORK = Path('/content/titanium_work')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
with zipfile.ZipFile(PACKAGE_ZIP) as z: z.extractall(WORK)
datasets = list(WORK.rglob('Titanium_Aging_RT_Resistivity_Dataset_Frozen_v1.1_2026-09-20.xlsx'))
assert len(datasets) == 1, f'Expected one dataset, found {len(datasets)}'
DATA = datasets[0]
ROOT = DATA.parent.parent
CODE = ROOT / 'code'
print('Package root:', ROOT)
print('Dataset:', DATA)


In [ ]:
%pip install -q -r "{CODE / 'requirements.txt'}"

## 1. Main resistivity and hardness model comparison

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(CODE/'main_ml'/'train.py'), '--input', str(DATA)], check=True)

## 2. XGBoost and SHAP

In [ ]:
subprocess.run([sys.executable, str(CODE/'main_ml'/'complete_xgb_shap.py')], check=True)

## 3. Paired hardness analysis with measured resistivity

In [ ]:
subprocess.run([sys.executable, str(CODE/'paired_ml'/'run.py'), str(DATA)], check=True)

## 4. View and download the new results
The repository’s `results/` folder contains the reference results. The following cell collects this run’s outputs into a separate folder and checks the scores against those references.


In [ ]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image

OUTPUT = WORK / 'Titanium_ML_New_Results'
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir()
sources = {
    'resistivity': CODE / 'ml_completed_v1' / 'resistivity',
    'hardness': CODE / 'ml_completed_v1' / 'hardness',
    'paired_hardness': CODE / 'paired_ml',
}
summary = []
for task, source in sources.items():
    assert (source / 'comparison.csv').is_file(), f'Missing results: {task}'
    destination = OUTPUT / task
    destination.mkdir()
    for path in source.iterdir():
        if path.is_file() and path.suffix in {'.csv', '.json', '.png', '.svg', '.joblib'}:
            shutil.copy2(path, destination / path.name)
    table = pd.read_csv(source / 'comparison.csv')
    if task == 'paired_hardness':
        chosen = json.loads((source / 'summary.json').read_text())['selected_tuned']
        for row in chosen:
            summary.append({'Analysis': task + '_' + row['variant'], 'Model': row['model'],
                            **{k: row[k] for k in ['CV_RMSE', 'test_R2', 'test_RMSE', 'test_MAE']}})
    else:
        row = table.loc[table['selected']].iloc[0]
        summary.append({'Analysis': task, 'Model': row['model'],
                        **{k: row[k] for k in ['CV_RMSE', 'test_R2', 'test_RMSE', 'test_MAE']}})
    reference = pd.read_csv(ROOT / 'results' / task / 'comparison.csv')
    keys = ['variant', 'model'] if 'variant' in table else ['model']
    metrics = ['CV_RMSE', 'train_R2', 'train_RMSE', 'train_MAE', 'test_R2', 'test_RMSE', 'test_MAE']
    current = table.set_index(keys).sort_index()[metrics]
    expected = reference.set_index(keys).sort_index()[metrics]
    matches = pd.DataFrame(
        np.isclose(current, expected, rtol=1e-6, atol=1e-8),
        index=current.index, columns=metrics,
    )
    verification = current.add_suffix('_current').join(expected.add_suffix('_reference'))
    verification = verification.join(matches.add_suffix('_matches'))
    verification.to_csv(destination / 'reference_comparison.csv')
    # KNN baseline scores can differ between execution environments.
    # Keep strict verification for every other model, including all selected models.
    strict_rows = current.index != 'KNN' if task != 'paired_hardness' else np.ones(len(current), dtype=bool)
    pd.testing.assert_frame_equal(
        current.loc[strict_rows], expected.loc[strict_rows],
        check_exact=False, rtol=1e-6, atol=1e-8,
    )
    if matches.to_numpy().all():
        print(task + ': all scores match the reference results.')
    else:
        print(task + ': selected models and non-KNN scores match; KNN baseline differs.')
        display(verification.loc[~matches.all(axis=1)])


scores = pd.DataFrame(summary)
scores.to_csv(OUTPUT / 'Summary.csv', index=False)
display(scores.round(4))
for task, filename in [('resistivity', 'prediction.png'), ('hardness', 'prediction.png'),
                       ('paired_hardness', 'paired_prediction.png')]:
    display(Image(filename=str(OUTPUT / task / filename), width=850))
(OUTPUT / 'Read_me.txt').write_text(
    'These files were generated by this run. Start with Summary.csv, then open the three analysis folders. '
    'CSV tables open in Excel; PNG files are figures; joblib files are fitted Python models. '
    'The original repository results were used only for the reproducibility check, not model fitting.\n'
)
archive = shutil.make_archive(str(WORK / 'Titanium_ML_New_Results'), 'zip', OUTPUT)
print('New results:', archive)
files.download(archive)
